# Source-14 × exact-UFE-12 overlap audit v1.1

This post-hoc diagnostic measures every pair between the 14 released source features and the 12 exact full-edge UFE dimensions. It uses only the fixed source training and validation masks, reports all 168 pairs, preserves unnamed edge attributes as attribute 0 and attribute 1, and does not access test features or labels. Candidate flags and rankings exclude observations where both features are zero; pairs containing a constant feature are reported separately and never flagged as overlaps. Results are descriptive and cannot establish identical upstream construction or causality.


In [ ]:
# Setup and frozen protocol verification
import gc
import hashlib
import json
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    import torch_geometric
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
    import torch_geometric

from scipy.stats import pearsonr, spearmanr
from torch_geometric.data import HeteroData

PROTOCOL_TEXT = "{\n  \"protocol_id\": \"anonymous-source-ufe-overlap-audit-v1.1\",\n  \"status\": \"AMENDED AND REFROZEN BEFORE EXECUTION OR OVERLAP RESULTS\",\n  \"frozen_utc\": \"2026-09-28T16:26:35Z\",\n  \"amendment\": {\n    \"previous_protocol_id\": \"anonymous-source-ufe-overlap-audit-v1\",\n    \"previous_protocol_sha256\": \"c3ee39528611539e2f66f8e1c4a0fefecea4d7f4f8ff45d5eebde997ae080f18\",\n    \"previous_notebook_sha256\": \"b7dea6604ed905cffc3a4dc97b5e6b19cfa92dee1f533f23f94b8ee431392451\",\n    \"reason\": \"Prevent jointly zero or constant columns from creating false overlap flags or inflated rankings. Candidate and near-duplicate decisions now use equality excluding both-zero observations, and constant pairs are reported separately.\",\n    \"execution_started_before_amendment\": false,\n    \"overlap_results_observed_before_amendment\": false,\n    \"raw_pairwise_statistics_changed\": false\n  },\n  \"scientific_role\": \"post-hoc exploratory provenance and overlap audit\",\n  \"dataset\": \"HEIG-Phish released PyG artifact\",\n  \"source_data_sha256\": \"8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8\",\n  \"target_node_type\": \"EOA\",\n  \"scope\": {\n    \"included\": \"the fixed source training and validation masks only\",\n    \"expected_nodes\": 2210,\n    \"test_features_used\": false,\n    \"test_labels_used\": false,\n    \"claim_boundary\": \"descriptive overlap audit only; not a predefined hypothesis test and not evidence of causal or independent behavioral information\"\n  },\n  \"source_features\": [\n    \"bal_c\",\n    \"V_inv_c\",\n    \"V_return_c\",\n    \"N_inv_c\",\n    \"N_return_c\",\n    \"V_meaninv_c\",\n    \"V_meanreturn_c\",\n    \"bal_t\",\n    \"V_inv_t\",\n    \"V_return_t\",\n    \"N_inv_t\",\n    \"N_return_t\",\n    \"V_meaninv_t\",\n    \"V_meanreturn_t\"\n  ],\n  \"ufe_definition\": {\n    \"relation_types\": [\"call\", \"trans\"],\n    \"directions\": [\"in\", \"out\"],\n    \"aggregates\": [\"incident edge-record count\", \"sum of released edge attribute 0\", \"sum of released edge attribute 1\"],\n    \"uses_labels\": false,\n    \"uses_exact_full_edges\": true,\n    \"transform_for_modeling_elsewhere\": \"log1p\",\n    \"terminology_note\": \"The two edge-attribute columns have no verified semantic headers in the released PyG artifact; this audit therefore calls them attribute 0 and attribute 1. The established project alias trans_in_sum corresponds to trans_in_edge_attr1_sum.\"\n  },\n  \"pairwise_design\": {\n    \"pairs\": 168,\n    \"raw_equality_rule\": \"numpy.isclose with rtol 1e-4 and atol 1e-8\",\n    \"pearson_transform\": \"signed_log1p(x) = sign(x) * log1p(abs(x)) applied independently to both raw columns\",\n    \"spearman_transform\": \"raw ranks\",\n    \"label_stratification\": [0, 1],\n    \"constant_feature_handling\": \"undefined correlations are recorded as null rather than zero\"\n  },\n  \"candidate_thresholds\": {\n    \"absolute_pearson_signed_log_at_least\": 0.8,\n    \"absolute_spearman_raw_at_least\": 0.8,\n    \"equal_or_close_excluding_both_zero_at_least\": 0.4,\n    \"near_duplicate_equal_or_close_excluding_both_zero_at_least\": 0.95,\n    \"flagging_rule\": \"candidate and near-duplicate flags use equal-or-close excluding both-zero observations; pairs containing a constant column are never flagged and are reported separately\"\n  },\n  \"planned_outputs\": [\n    \"feature_summary.csv\",\n    \"pairwise_overlap_14x12.csv\",\n    \"pairwise_overlap_by_label.csv\",\n    \"pearson_signed_log_matrix.csv\",\n    \"spearman_raw_matrix.csv\",\n    \"equal_or_close_matrix.csv\",\n    \"equal_or_close_excluding_both_zero_matrix.csv\",\n    \"candidate_overlaps.csv\",\n    \"constant_or_structurally_zero_pairs.csv\",\n    \"top_match_per_source_feature.csv\",\n    \"top_match_per_ufe_feature.csv\",\n    \"four heatmap PNG files\",\n    \"final_summary.json\"\n  ],\n  \"reporting_commitment\": \"Report all 168 pairs and all constant or structurally zero dimensions, including results that weaken the UFE novelty interpretation.\"\n}\n"
PROTOCOL = json.loads(PROTOCOL_TEXT)
PROTOCOL_SHA256 = "bbb4bb8359a438a90e5b74ec95fefbf4cbf09001fe8d602a5bf01fc713a4e42c"
EXPECTED_DATA_SHA256 = PROTOCOL["source_data_sha256"]
RTOL = 1e-4
ATOL = 1e-8

SOURCE_NAMES = PROTOCOL["source_features"]
UFE_NAMES = [
    "call_in_edge_records", "call_out_edge_records",
    "call_in_edge_attr0_sum", "call_out_edge_attr0_sum",
    "call_in_edge_attr1_sum", "call_out_edge_attr1_sum",
    "trans_in_edge_records", "trans_out_edge_records",
    "trans_in_edge_attr0_sum", "trans_out_edge_attr0_sum",
    "trans_in_edge_attr1_sum", "trans_out_edge_attr1_sum",
]
UFE_ALIASES = {
    "call_in_edge_records": "call_in_rows",
    "call_out_edge_records": "call_out_rows",
    "call_in_edge_attr0_sum": "call_in_count (unverified semantic alias)",
    "call_out_edge_attr0_sum": "call_out_count (unverified semantic alias)",
    "call_in_edge_attr1_sum": "call_in_sum",
    "call_out_edge_attr1_sum": "call_out_sum",
    "trans_in_edge_records": "trans_in_rows",
    "trans_out_edge_records": "trans_out_rows",
    "trans_in_edge_attr0_sum": "trans_in_count (unverified semantic alias)",
    "trans_out_edge_attr0_sum": "trans_out_count (unverified semantic alias)",
    "trans_in_edge_attr1_sum": "trans_in_sum",
    "trans_out_edge_attr1_sum": "trans_out_sum",
}

assert len(SOURCE_NAMES) == 14 and len(UFE_NAMES) == 12
assert hashlib.sha256(PROTOCOL_TEXT.encode("utf-8")).hexdigest() == PROTOCOL_SHA256

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def sha256_array(values, dtype):
    return hashlib.sha256(np.ascontiguousarray(values, dtype=dtype).tobytes()).hexdigest()

data_candidates = [
    path for path in Path("/kaggle/input").rglob("data.pt")
    if "/phish/processed/data.pt" in str(path).replace("\\", "/").lower()
]
assert len(data_candidates) == 1, [str(path) for path in data_candidates]
data_path = data_candidates[0]
data_sha256 = sha256_file(data_path)
assert data_sha256 == EXPECTED_DATA_SHA256, {"expected": EXPECTED_DATA_SHA256, "observed": data_sha256}

loaded = torch.load(data_path, map_location="cpu", weights_only=False)
data = loaded[0] if isinstance(loaded, tuple) and len(loaded) == 2 else loaded
assert isinstance(data, HeteroData), type(data)
del loaded
gc.collect()

target_store = data["EOA"]
assert target_store.x.ndim == 2 and target_store.x.shape[1] == 14
assert (int(target_store.train_mask.sum()), int(target_store.val_mask.sum()), int(target_store.test_mask.sum())) == (1657, 553, 553)
assert not bool((target_store.train_mask & target_store.val_mask).any())
assert not bool((target_store.train_mask & target_store.test_mask).any())
assert not bool((target_store.val_mask & target_store.test_mask).any())

train_ids = torch.where(target_store.train_mask)[0].cpu().numpy().astype(np.int64)
validation_ids = torch.where(target_store.val_mask)[0].cpu().numpy().astype(np.int64)
development_ids = np.sort(np.concatenate([train_ids, validation_ids]))
assert len(development_ids) == PROTOCOL["scope"]["expected_nodes"]
assert len(np.unique(development_ids)) == len(development_ids)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("/kaggle/working") / f"source_ufe_overlap_audit_v1_1_{timestamp}"
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / "protocol.json").write_text(PROTOCOL_TEXT, encoding="utf-8")
assert sha256_file(run_dir / "protocol.json") == PROTOCOL_SHA256

print("Protocol SHA256:", PROTOCOL_SHA256)
print("Data SHA256:", data_sha256)
print("Development nodes:", len(development_ids))
print("Output:", run_dir)


In [ ]:
# Materialize exact full-edge UFE for train and validation nodes only
RELATION_TO_INDEX = {"call": 0, "trans": 1}

def exact_full_edge_ufe_raw(node_ids, chunk_size=2_000_000):
    node_ids = np.asarray(node_ids, dtype=np.int64)
    lookup = torch.full((target_store.num_nodes,), -1, dtype=torch.long)
    lookup[torch.from_numpy(node_ids)] = torch.arange(len(node_ids), dtype=torch.long)
    result = torch.zeros((len(node_ids), len(UFE_NAMES)), dtype=torch.float64)

    for edge_type in data.edge_types:
        source_type, relation, destination_type = edge_type
        assert relation in RELATION_TO_INDEX, relation
        store = data[edge_type]
        assert hasattr(store, "edge_attr") and store.edge_attr.ndim == 2 and store.edge_attr.shape[1] >= 2
        edge_count = int(store.edge_index.shape[1])
        base = RELATION_TO_INDEX[relation] * 6

        for start in range(0, edge_count, chunk_size):
            stop = min(start + chunk_size, edge_count)
            attrs = torch.nan_to_num(
                store.edge_attr[start:stop, :2].to(torch.float64),
                nan=0.0, posinf=0.0, neginf=0.0,
            ).clamp_min_(0.0)

            if destination_type == "EOA":
                positions = lookup[store.edge_index[1, start:stop]]
                keep = positions >= 0
                if bool(keep.any()):
                    selected = positions[keep]
                    result[:, base + 0].index_add_(0, selected, torch.ones(len(selected), dtype=torch.float64))
                    result[:, base + 2].index_add_(0, selected, attrs[keep, 0])
                    result[:, base + 4].index_add_(0, selected, attrs[keep, 1])

            if source_type == "EOA":
                positions = lookup[store.edge_index[0, start:stop]]
                keep = positions >= 0
                if bool(keep.any()):
                    selected = positions[keep]
                    result[:, base + 1].index_add_(0, selected, torch.ones(len(selected), dtype=torch.float64))
                    result[:, base + 3].index_add_(0, selected, attrs[keep, 0])
                    result[:, base + 5].index_add_(0, selected, attrs[keep, 1])

    del lookup
    return result.numpy()

started = time.perf_counter()
ufe_raw = exact_full_edge_ufe_raw(development_ids)
materialization_seconds = time.perf_counter() - started
source_raw = target_store.x[torch.from_numpy(development_ids)].cpu().numpy().astype(np.float64, copy=True)
development_y = target_store.y[torch.from_numpy(development_ids)].cpu().numpy().astype(np.int64, copy=True)

assert source_raw.shape == (2210, 14)
assert ufe_raw.shape == (2210, 12)
assert set(np.unique(development_y)) == {0, 1}
assert np.isfinite(source_raw).all() and np.isfinite(ufe_raw).all()
assert np.allclose(ufe_raw[:, [0, 2, 4]], 0.0), "Expected EOA call-in structural zeros were not zero"

print("Exact UFE materialization seconds:", materialization_seconds)
print("Source shape:", source_raw.shape, "UFE shape:", ufe_raw.shape)


In [ ]:
# Compute all 14 x 12 pairwise overlap statistics
def signed_log1p(values):
    values = np.asarray(values, dtype=np.float64)
    return np.sign(values) * np.log1p(np.abs(values))

def is_constant(values):
    values = np.asarray(values, dtype=np.float64)
    return bool(np.allclose(values, values[0], rtol=0.0, atol=ATOL))

def safe_pearson(x, y):
    if is_constant(x) or is_constant(y):
        return np.nan
    return float(pearsonr(x, y).statistic)

def safe_spearman(x, y):
    if is_constant(x) or is_constant(y):
        return np.nan
    return float(spearmanr(x, y).statistic)

def feature_summary(block, names, matrix):
    rows = []
    for index, name in enumerate(names):
        values = matrix[:, index]
        rows.append({
            "block": block,
            "feature_index": index,
            "feature": name,
            "project_alias": UFE_ALIASES.get(name, ""),
            "nodes": len(values),
            "constant": is_constant(values),
            "zero_fraction": float(np.isclose(values, 0.0, rtol=0.0, atol=ATOL).mean()),
            "unique_values": int(len(np.unique(values))),
            "minimum": float(values.min()),
            "median": float(np.median(values)),
            "mean": float(values.mean()),
            "sd": float(values.std(ddof=1)),
            "maximum": float(values.max()),
            "zero_fraction_normal": float(np.isclose(values[development_y == 0], 0.0, rtol=0.0, atol=ATOL).mean()),
            "zero_fraction_phishing": float(np.isclose(values[development_y == 1], 0.0, rtol=0.0, atol=ATOL).mean()),
        })
    return rows

summary_rows = feature_summary("source14", SOURCE_NAMES, source_raw)
summary_rows += feature_summary("ufe12", UFE_NAMES, ufe_raw)
feature_summary_df = pd.DataFrame(summary_rows)
feature_summary_df.to_csv(run_dir / "feature_summary.csv", index=False)

pair_rows = []
label_rows = []
for source_index, source_name in enumerate(SOURCE_NAMES):
    source_values = source_raw[:, source_index]
    source_log = signed_log1p(source_values)
    for ufe_index, ufe_name in enumerate(UFE_NAMES):
        ufe_values = ufe_raw[:, ufe_index]
        ufe_log = signed_log1p(ufe_values)
        close = np.isclose(source_values, ufe_values, rtol=RTOL, atol=ATOL)
        source_zero = np.isclose(source_values, 0.0, rtol=0.0, atol=ATOL)
        ufe_zero = np.isclose(ufe_values, 0.0, rtol=0.0, atol=ATOL)
        both_zero = source_zero & ufe_zero
        non_both_zero = ~both_zero
        pearson_value = safe_pearson(source_log, ufe_log)
        spearman_value = safe_spearman(source_values, ufe_values)
        equal_nonzero = float(close[non_both_zero].mean()) if bool(non_both_zero.any()) else np.nan
        abs_differences = np.abs(source_values - ufe_values)
        pair_has_constant = is_constant(source_values) or is_constant(ufe_values)
        candidate = (not pair_has_constant) and (
            (np.isfinite(pearson_value) and abs(pearson_value) >= PROTOCOL["candidate_thresholds"]["absolute_pearson_signed_log_at_least"])
            or (np.isfinite(spearman_value) and abs(spearman_value) >= PROTOCOL["candidate_thresholds"]["absolute_spearman_raw_at_least"])
            or (
                np.isfinite(equal_nonzero)
                and equal_nonzero >= PROTOCOL["candidate_thresholds"]["equal_or_close_excluding_both_zero_at_least"]
            )
        )
        near_duplicate = (not pair_has_constant) and np.isfinite(equal_nonzero) and (
            equal_nonzero
            >= PROTOCOL["candidate_thresholds"]["near_duplicate_equal_or_close_excluding_both_zero_at_least"]
        )
        overlap_components = []
        if not pair_has_constant:
            if np.isfinite(equal_nonzero):
                overlap_components.append(equal_nonzero)
            if np.isfinite(pearson_value):
                overlap_components.append(abs(pearson_value))
            if np.isfinite(spearman_value):
                overlap_components.append(abs(spearman_value))
        ranking_score = float(max(overlap_components)) if overlap_components else np.nan
        pair_class = (
            "constant_or_structurally_zero_pair" if pair_has_constant
            else "near_duplicate_candidate" if near_duplicate
            else "candidate_overlap" if candidate
            else "not_flagged"
        )

        pair_rows.append({
            "source_index": source_index,
            "source_feature": source_name,
            "ufe_index": ufe_index,
            "ufe_feature": ufe_name,
            "ufe_project_alias": UFE_ALIASES[ufe_name],
            "pearson_signed_log1p": pearson_value,
            "spearman_raw": spearman_value,
            "equal_or_close_fraction": float(close.mean()),
            "equal_or_close_excluding_both_zero": equal_nonzero,
            "both_zero_fraction": float(both_zero.mean()),
            "source_zero_fraction": float(source_zero.mean()),
            "ufe_zero_fraction": float(ufe_zero.mean()),
            "median_absolute_raw_difference": float(np.median(abs_differences)),
            "mean_absolute_raw_difference": float(abs_differences.mean()),
            "contains_constant_feature": bool(pair_has_constant),
            "pair_class": pair_class,
            "candidate_overlap": bool(candidate),
            "near_duplicate": bool(near_duplicate),
            "ranking_score": ranking_score,
        })

        for label in (0, 1):
            mask = development_y == label
            label_source = source_values[mask]
            label_ufe = ufe_values[mask]
            label_close = close[mask]
            label_both_zero = both_zero[mask]
            label_non_both_zero = ~label_both_zero
            label_rows.append({
                "source_feature": source_name,
                "ufe_feature": ufe_name,
                "ufe_project_alias": UFE_ALIASES[ufe_name],
                "label": label,
                "nodes": int(mask.sum()),
                "pearson_signed_log1p": safe_pearson(signed_log1p(label_source), signed_log1p(label_ufe)),
                "spearman_raw": safe_spearman(label_source, label_ufe),
                "equal_or_close_fraction": float(label_close.mean()),
                "equal_or_close_excluding_both_zero": (
                    float(label_close[label_non_both_zero].mean()) if bool(label_non_both_zero.any()) else np.nan
                ),
                "both_zero_fraction": float(label_both_zero.mean()),
            })

pairs = pd.DataFrame(pair_rows)
by_label = pd.DataFrame(label_rows)
assert len(pairs) == 168 and len(by_label) == 336
pairs.to_csv(run_dir / "pairwise_overlap_14x12.csv", index=False)
by_label.to_csv(run_dir / "pairwise_overlap_by_label.csv", index=False)

candidates = pairs[pairs.candidate_overlap].sort_values(
    ["ranking_score", "equal_or_close_excluding_both_zero"], ascending=[False, False]
)
candidates.to_csv(run_dir / "candidate_overlaps.csv", index=False)

constant_pairs = pairs[pairs.contains_constant_feature].sort_values(
    ["source_feature", "ufe_feature"]
)
constant_pairs.to_csv(run_dir / "constant_or_structurally_zero_pairs.csv", index=False)

ranked_for_source = pairs.sort_values(
    ["source_feature", "ranking_score", "equal_or_close_excluding_both_zero"],
    ascending=[True, False, False], na_position="last",
)
ranked_for_ufe = pairs.sort_values(
    ["ufe_feature", "ranking_score", "equal_or_close_excluding_both_zero"],
    ascending=[True, False, False], na_position="last",
)
top_per_source = ranked_for_source.groupby("source_feature", sort=False, as_index=False).head(1)
top_per_ufe = ranked_for_ufe.groupby("ufe_feature", sort=False, as_index=False).head(1)
top_per_source.to_csv(run_dir / "top_match_per_source_feature.csv", index=False)
top_per_ufe.to_csv(run_dir / "top_match_per_ufe_feature.csv", index=False)

print("All pairs:", len(pairs), "Candidate overlaps:", len(candidates), "Constant pairs:", len(constant_pairs))
display(candidates.head(30))


In [ ]:
# Export matrices and readable heatmaps
def make_matrix(value_column):
    return pairs.pivot(index="source_feature", columns="ufe_feature", values=value_column).reindex(
        index=SOURCE_NAMES, columns=UFE_NAMES
    )

pearson_matrix = make_matrix("pearson_signed_log1p")
spearman_matrix = make_matrix("spearman_raw")
equality_matrix = make_matrix("equal_or_close_fraction")
equality_nonzero_matrix = make_matrix("equal_or_close_excluding_both_zero")

pearson_matrix.to_csv(run_dir / "pearson_signed_log_matrix.csv")
spearman_matrix.to_csv(run_dir / "spearman_raw_matrix.csv")
equality_matrix.to_csv(run_dir / "equal_or_close_matrix.csv")
equality_nonzero_matrix.to_csv(run_dir / "equal_or_close_excluding_both_zero_matrix.csv")

def save_heatmap(matrix, title, output_name, vmin, vmax, cmap, percentage=False):
    values = matrix.to_numpy(dtype=float)
    masked = np.ma.masked_invalid(values)
    fig, ax = plt.subplots(figsize=(18, 13))
    image = ax.imshow(masked, aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(np.arange(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=50, ha="right", fontsize=9)
    ax.set_yticks(np.arange(len(matrix.index)))
    ax.set_yticklabels(matrix.index, fontsize=10)
    ax.set_title(title, fontsize=15, pad=18)
    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            value = values[row, column]
            if np.isfinite(value):
                label = f"{100 * value:.0f}%" if percentage else f"{value:.2f}"
                ax.text(column, row, label, ha="center", va="center", fontsize=7)
            else:
                ax.text(column, row, "NA", ha="center", va="center", fontsize=7, color="gray")
    colorbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
    colorbar.ax.tick_params(labelsize=9)
    fig.tight_layout()
    fig.savefig(run_dir / output_name, dpi=220, bbox_inches="tight")
    plt.close(fig)

save_heatmap(
    pearson_matrix,
    "Source 14 vs exact UFE 12: Pearson correlation after signed log1p",
    "pearson_signed_log_heatmap.png", -1.0, 1.0, "coolwarm",
)
save_heatmap(
    spearman_matrix,
    "Source 14 vs exact UFE 12: Spearman correlation on raw values",
    "spearman_raw_heatmap.png", -1.0, 1.0, "coolwarm",
)
save_heatmap(
    equality_matrix,
    "Source 14 vs exact UFE 12: raw equal-or-close fraction (includes both-zero observations)",
    "equal_or_close_heatmap.png", 0.0, 1.0, "viridis", percentage=True,
)
save_heatmap(
    equality_nonzero_matrix,
    "Source 14 vs exact UFE 12: equal-or-close fraction excluding both-zero observations",
    "equal_or_close_excluding_both_zero_heatmap.png", 0.0, 1.0, "viridis", percentage=True,
)

print("Matrices and heatmaps saved.")


In [ ]:
# Freeze outputs and create the downloadable evidence ZIP
constant_source = feature_summary_df.query("block == 'source14' and constant == True")["feature"].tolist()
constant_ufe = feature_summary_df.query("block == 'ufe12' and constant == True")["feature"].tolist()

established_pair = pairs[
    (pairs.source_feature == "V_inv_t")
    & (pairs.ufe_feature == "trans_in_edge_attr1_sum")
].iloc[0].to_dict()

final_summary = {
    "status": "completed post-hoc train-and-validation-only 14-by-12 overlap audit",
    "protocol_sha256": PROTOCOL_SHA256,
    "source_data_sha256": data_sha256,
    "development_node_count": int(len(development_ids)),
    "normal_node_count": int((development_y == 0).sum()),
    "phishing_node_count": int((development_y == 1).sum()),
    "pair_count": int(len(pairs)),
    "candidate_overlap_count": int(len(candidates)),
    "constant_or_structurally_zero_pair_count": int(len(constant_pairs)),
    "constant_source_features": constant_source,
    "constant_ufe_features": constant_ufe,
    "established_trans_in_sum_vs_V_inv_t": established_pair,
    "development_ids_sha256": sha256_array(development_ids, np.int64),
    "source14_sha256": sha256_array(source_raw, np.float64),
    "ufe12_raw_sha256": sha256_array(ufe_raw, np.float64),
    "materialization_seconds": materialization_seconds,
    "test_labels_accessed": False,
    "test_features_sliced_or_analyzed": False,
    "interpretation": "Exploratory descriptive provenance audit. Correlation and equality do not prove identical upstream construction, causality, or leakage.",
}
(run_dir / "final_summary.json").write_text(json.dumps(final_summary, indent=2), encoding="utf-8")

manifest_rows = []
for output_path in sorted(path for path in run_dir.iterdir() if path.is_file()):
    manifest_rows.append({
        "file": output_path.name,
        "bytes": output_path.stat().st_size,
        "sha256": sha256_file(output_path),
    })
pd.DataFrame(manifest_rows).to_csv(run_dir / "file_manifest.csv", index=False)

zip_path = Path(shutil.make_archive(str(run_dir), "zip", run_dir))
print(json.dumps(final_summary, indent=2))
print("DOWNLOAD:", zip_path)
